##02-2. 지하철 승하차

* API로 받은 **원본 그대로** PostgreSQL 테이블에 만들고 저장 (파일로 저장하지 않음)
* 아래 셀은 앞의 셀이 먼저 실행되어 있어야 함
  * `conn`, `ensure_conn()`, `save_df()` : DB 연결, DB 저장 공통함수
  * `fetch_all_rows()`, `check_api()` : 서울 열린데이터광장 API 수집 함수
  * `SEOUL_API_KEY`, `START_DATE`, `END_DATE` : 인증키, 수집 기간
* 컬럼은 API 응답 컬럼명을 소문자로 바꿔 전부 TEXT로 저장

###02-2-1. 지하철 역 정보

서울시 역사마스터 정보 (`subwayStationMaster`) > 출처: 서울 열린데이터광장

* 역ID·역명·호선·위도·경도를 확인할 수 있는 원본 테이블 `RAW_SUBWAY_STATION_MASTER` 생성
* 환승역 묶기, 좌표 검증은 저장 이후 단계에서 진행

In [ ]:
# ============================================================
# [지하철 공통] 테이블 존재 여부 확인
#   - save_df는 새 테이블을 만들 때 이미 있으면 오류를 내므로, 저장 전에 미리 확인
# ============================================================
def table_exists(table_name):
    with conn.cursor() as cur:
        cur.execute("SELECT to_regclass(%s::text)", [table_name.lower()])
        exists = cur.fetchone()[0] is not None
    conn.rollback()                                 # 확인용 트랜잭션 정리
    return exists

In [ ]:
# api 정상접속 확인 : 첫 5행만 조회
df_subway_master_check = check_api(service="subwayStationMaster")

In [ ]:
# ============================================================
# [역 정보 DB 저장] API 전체 수집 → RAW_SUBWAY_STATION_MASTER 생성 + 저장
# ============================================================
MASTER_TABLE = "RAW_SUBWAY_STATION_MASTER"

if table_exists(MASTER_TABLE):
    print(f"{MASTER_TABLE}: 이미 있는 테이블이라 저장하지 않습니다.")
    print("다시 넣으려면 DROP TABLE 후 이 셀을 다시 실행하세요.")
else:
    raw_subway_master = fetch_all_rows("subwayStationMaster")   # 1,000행씩 끝까지 수집

    print("역사마스터:", raw_subway_master.shape)
    print("원본 컬럼:", raw_subway_master.columns.tolist())
    display(raw_subway_master.head())

    if raw_subway_master.empty:
        raise RuntimeError("수집된 역사마스터 데이터가 없어 저장하지 않습니다.")

    save_df(raw_subway_master, MASTER_TABLE, conn)

###02-2-2. 승하차 정보

교통카드 정산시스템 시간대별 승하차 (`CardSubwayTime`, OA-12252) > 출처: 서울 열린데이터광장

* 조건값은 사용월(YYYYMM). 2021-01 ~ 2026-09, 월별로 API 호출 → `RAW_SUBWAY_CARD_TIME`에 저장
* 한 달 단위로 **데이터 저장과 이력 기록을 한 번에 커밋** → 중간에 끊겨도 중복/누락 없음
* 월별 진행 상황은 `SUBWAY_CARD_TIME_LOAD_LOG`에 기록 (`DONE` 완료 / `EMPTY` 빈 응답)
* 다시 실행하면 완료된 달은 건너뛰고 이어서 진행

In [ ]:
# api 정상접속 확인 : 2021년 1월 첫 5행만 조회
df_subway_time_check = check_api(service="CardSubwayTime", date_parameter="202101")

In [ ]:
# ============================================================
# [승하차 DB 저장 1] 설정값 + 월별 적재 이력 테이블 관리
#   status = 'DONE'(저장 완료) | 'EMPTY'(API 빈 응답, 저장 안 함)
# ============================================================
SUBWAY_SERVICE = "CardSubwayTime"
SUBWAY_TABLE   = "RAW_SUBWAY_CARD_TIME"            # 원본 데이터 테이블
SUBWAY_LOG     = "SUBWAY_CARD_TIME_LOAD_LOG"       # 월별 적재 이력 테이블
MAX_RETRIES    = 3


def ensure_subway_log():
    ensure_conn()
    with conn.cursor() as cur:
        cur.execute(f"""CREATE TABLE IF NOT EXISTS {SUBWAY_LOG} (
            use_month TEXT PRIMARY KEY,
            row_cnt   BIGINT,
            status    TEXT,
            loaded_at TIMESTAMP DEFAULT now())""")
    conn.commit()


def get_subway_log():
    """{사용월: 상태}"""
    with conn.cursor() as cur:
        cur.execute(f"SELECT use_month, status FROM {SUBWAY_LOG}")
        log = {r[0]: r[1] for r in cur.fetchall()}
    conn.rollback()
    return log


def set_subway_log(use_month, row_cnt, status, commit=True):
    """월별 진행 상황 기록 (있으면 갱신, 없으면 추가)"""
    with conn.cursor() as cur:
        cur.execute(f"""
            INSERT INTO {SUBWAY_LOG} (use_month, row_cnt, status) VALUES (%s, %s, %s)
            ON CONFLICT (use_month) DO UPDATE
            SET row_cnt = EXCLUDED.row_cnt, status = EXCLUDED.status, loaded_at = now()""",
            [use_month, row_cnt, status])
    if commit:
        conn.commit()

In [ ]:
# ============================================================
# [승하차 DB 저장 2] 한 달 수집 → 저장
#   - 통신/JSON 응답 실패는 최대 3회 재시도, 인증키/API 오류는 그대로 중단
#   - 수집이 끝난 뒤에만 DB에 저장하고, 데이터 + 이력을 한 트랜잭션으로 커밋
# ============================================================
def fetch_month(use_month):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return fetch_all_rows(SUBWAY_SERVICE, use_month)
        except RuntimeError as error:
            if "행 요청 실패" not in str(error):
                raise
            print(f"[요청 실패] {attempt}/{MAX_RETRIES}회")
            if attempt == MAX_RETRIES:
                raise
            wait_seconds = attempt * 3
            print(f"{wait_seconds}초 후 {use_month}를 다시 조회합니다.")
            time.sleep(wait_seconds)


def load_subway_month(use_month):
    df_month = fetch_month(use_month)
    print(f"수신 건수: {len(df_month):,}행")

    ensure_conn()                                       # 수집하는 동안 끊겼으면 재접속
    try:
        if df_month.empty:
            # 빈 응답을 '승하차 0명'으로 확정하지 않고 기록만 남김
            set_subway_log(use_month, 0, "EMPTY", commit=False)
        else:
            save_df(df_month, SUBWAY_TABLE, conn, mode="INSERT", commit=False)  # 커밋 보류
            set_subway_log(use_month, len(df_month), "DONE", commit=False)
        conn.commit()                                   # 데이터 + 이력 동시 커밋
    except Exception:
        conn.rollback()
        raise
    return len(df_month)

In [ ]:
# ============================================================
# [승하차 DB 저장 3] 기간 전체 반복 적재 (월별 반복문)
#   - 완료(DONE)된 달은 건너뜀 → 다시 실행하면 실패한 달부터 이어서 진행
#   - max_months 로 일부만 먼저 시험 가능 (예: 1)
# ============================================================
def load_subway_time(max_months=None, stop_on_error=True):
    ensure_conn()
    ensure_subway_log()

    all_months = [m.strftime("%Y%m") for m in pd.period_range(START_DATE, END_DATE, freq="M")]
    log = get_subway_log()
    todo = [ym for ym in all_months if log.get(ym) != "DONE"]
    if max_months:
        todo = todo[:max_months]

    n_done = sum(1 for ym in all_months if log.get(ym) == "DONE")
    print(f"전체 {len(all_months)}개월 / 완료 {n_done}개월 / 이번 처리 {len(todo)}개월")

    failed = {}
    for i, ym in enumerate(todo, 1):                    # ← 월별 반복문
        print(f"\n[{i}/{len(todo)}] 사용월: {ym}")
        t0 = time.time()
        try:
            n = load_subway_month(ym)
            print(f"   ✔ 완료: {n:,}행 / {time.time()-t0:,.0f}초")
        except Exception as e:
            failed[ym] = str(e)
            print(f"   ⚠ 실패: {e}")
            if stop_on_error:
                print("   → 중단합니다. 원인 해결 후 다시 실행하면 이 달부터 이어서 진행합니다.")
                break
        time.sleep(0.2)

    print(f"\n처리 종료: 실패 {len(failed)}개", failed if failed else "")
    return failed

In [ ]:
# 먼저 1개월만 시험 저장해서 테이블이 잘 만들어지는지 확인
failed = load_subway_time(max_months=1)

In [ ]:
# 전체 기간 저장
failed = load_subway_time()

In [ ]:
# ============================================================
# [승하차 DB 저장 4] 저장 결과 확인
# ============================================================
conn.rollback()      # 오류로 중단된 트랜잭션이 남아 있으면 정리 (필수)

with conn.cursor() as cur:
    # 1) 적재 이력: 상태별 개월 수와 로그상 총 행 수
    cur.execute(f"SELECT status, COUNT(*), SUM(row_cnt) FROM {SUBWAY_LOG} GROUP BY status ORDER BY status")
    for status, n_months, n_rows in cur.fetchall():
        print(f"{status}: {n_months}개월 / {int(n_rows or 0):,}행")

    # 2) 실제 행 수가 이력과 맞는지
    cur.execute(f"SELECT COALESCE(SUM(row_cnt), 0) FROM {SUBWAY_LOG} WHERE status = 'DONE'")
    log_rows = int(cur.fetchone()[0])
    cur.execute(f"SELECT COUNT(*) FROM {SUBWAY_TABLE}")
    real_rows = cur.fetchone()[0]
    print(f"실제 행 수: {real_rows:,}  →  로그와 {'일치 ✔' if real_rows == log_rows else '불일치 ⚠'}")

    # 3) 컬럼과 샘플 확인
    cur.execute(f"SELECT * FROM {SUBWAY_TABLE} LIMIT 3")
    print("컬럼:", [d[0] for d in cur.description])
    for r in cur.fetchall():
        print(" ", r)
conn.rollback()